<a href="https://colab.research.google.com/github/dantruongnv/AI-Email-Spam-Detector/blob/main/AI_Spam_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📧 AI Email Spam Detector with Gmail API
Dự án phân loại và tự động xử lý email rác (Spam) trong Gmail sử dụng Machine Learning (Naive Bayes + TF-IDF).

## Bước 1: Cài đặt các thư viện cần thiết

In [ ]:
!pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib scikit-learn pandas

## Bước 2: Tải lên file credentials.json
*Lưu ý: File credentials.json lấy từ Google Cloud Console (OAuth 2.0 Client ID).*

In [ ]:
import os
from google.colab import files

if not os.path.exists('credentials.json'):
    print("Hãy chọn file credentials.json từ máy tính của bạn:")
    uploaded = files.upload()
else:
    print("✅ File credentials.json đã tồn tại trên môi trường Colab.")

## Bước 3: Xác thực OAuth 2.0 & Lấy thư chưa đọc từ Gmail API

In [ ]:
import os.path
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials

SCOPES = ['https://www.googleapis.com/auth/gmail.modify']

def get_unread_emails():
    creds = None
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            flow.redirect_uri = 'https://localhost'
            auth_url, _ = flow.authorization_url(prompt='consent')

            print("1. Click vào liên kết sau để cấp quyền Google:")
            print(auth_url)
            print("\n2. Sau khi bấm 'Cho phép', trình duyệt chuyển sang trang 'localhost' báo lỗi không tìm thấy trang.")
            print("   -> COPY TOÀN BỘ URL TRÊN THANH ĐỊA CHỈ DÁN VÀO BÊN DƯỚI.")

            redirect_response = input("\n3. Dán toàn bộ URL trang lỗi vào đây: ")
            flow.fetch_token(authorization_response=redirect_response)
            creds = flow.credentials

        with open('token.json', 'w') as token:
            token.write(creds.to_json())

    service = build('gmail', 'v1', credentials=creds)
    results = service.users().messages().list(userId='me', q='is:unread', maxResults=5).execute()
    messages = results.get('messages', [])

    emails_data = []
    if not messages:
        print("\n[THÔNG BÁO]: Không có email nào chưa đọc trong Inbox.")
    else:
        print(f"\n[THÀNH CÔNG]: Đã kết nối Gmail và lấy {len(messages)} email chưa đọc!")
        for message in messages:
            msg = service.users().messages().get(userId='me', id=message['id']).execute()
            snippet = msg.get('snippet', '')
            emails_data.append({'id': message['id'], 'snippet': snippet})

    return service, emails_data

service, unread_emails = get_unread_emails()

## Bước 4: Huấn luyện mô hình AI & Phân loại Email thực tế

In [ ]:
import os
import urllib.request
import zipfile
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Tải bộ dữ liệu SMS Spam Collection Dataset
print("1. Đang tải bộ dữ liệu Spam Dataset...")
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "smsspamcollection.zip"

if not os.path.exists("SMSSpamCollection"):
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("   -> Tải và giải nén Dataset thành công!")

# 2. Tiền xử lý dữ liệu
df = pd.read_csv('SMSSpamCollection', sep='\t', names=['label', 'text'])
print(f"   -> Tổng số mẫu huấn luyện: {len(df)} dòng")

X = df['text']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Trích xuất đặc trưng & Huấn luyện Naive Bayes
print("\n2. Đang huấn luyện mô hình AI (Naive Bayes + TF-IDF)...")
vectorizer = TfidfVectorizer(stop_words='english', lowercase=True)
X_train_vec = vectorizer.fit_transform(X_train)

model = MultinomialNB()
model.fit(X_train_vec, y_train)

X_test_vec = vectorizer.transform(X_test)
acc = accuracy_score(y_test, model.predict(X_test_vec))
print(f"   -> Độ chính xác của mô hình trên tập kiểm thử: {acc * 100:.2f}%")

# 4. Dự đoán và gắn nhãn tự động trên Gmail
print("\n" + "=" * 60)
print("=== BẮT ĐẦU PHÂN LOẠI HÒM THƯ GMAIL CỦA BẠN BẰNG AI ===")
print("=" * 60)

if 'unread_emails' not in locals() or not unread_emails:
    print("[THÔNG BÁO]: Không tìm thấy thư mới chưa đọc trong Hộp thư đến.")
else:
    print(f"Tìm thấy {len(unread_emails)} email chưa đọc từ Hộp thư đến (Inbox).\n")
    for index, item in enumerate(unread_emails, 1):
        email_text = item['snippet']
        email_id = item['id']
        
        text_vector = vectorizer.transform([email_text])
        prediction = model.predict(text_vector)[0]
        confidence = model.predict_proba(text_vector).max() * 100
        
        print(f"[{index}] Nội dung email: '{email_text[:80]}...'")
        if prediction == 'spam':
            print(f"   -> AI ĐÁNH GIÁ: [ SPAM / THƯ RÁC ] (Độ tin cậy: {confidence:.1f}%)")
            service.users().messages().modify(
                userId='me',
                id=email_id,
                body={'addLabelIds': ['SPAM'], 'removeLabelIds': ['INBOX']}
            ).execute()
            print("   -> HÀNH ĐỘNG: Tự động chuyển thư vào thư mục SPAM!")
        else:
            print(f"   -> AI ĐÁNH GIÁ: [ HAM / THƯ THƯỜNG ] (Độ tin cậy: {confidence:.1f}%)")
            print("   -> HÀNH ĐỘNG: Giữ nguyên trong Hộp thư đến.")
        print("-" * 60)